# 06 - End-to-End ANPR Evaluation

Evaluate the fine-tuned recognizer per image, analyze character confusions, calibrate a decision gate on validation data, integrate YOLO plate detection, test uploaded vehicle images, and create an active-learning queue.

In [ ]:
import re
from difflib import SequenceMatcher
from pathlib import Path
from time import perf_counter

import cv2
import ipywidgets as widgets
import numpy as np
import pandas as pd
import torch  # Load before Paddle on Windows.
import paddle
from IPython.display import Image as DisplayImage, clear_output, display
from paddleocr import TextRecognition
from ultralytics import YOLO

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'ml').is_dir() and (candidate / 'data').is_dir():
            return candidate
    raise FileNotFoundError('Project root not found.')

PROJECT_ROOT = find_project_root(Path.cwd().resolve())
DATA_ROOT = PROJECT_ROOT / 'data/processed/plate_ocr'
MANIFEST_ROOT = DATA_ROOT / 'manifests'
ANNOTATION_PATH = DATA_ROOT / 'annotations.csv'
RECOGNIZER_DIR = PROJECT_ROOT / 'ml/experiments/paddleocr_plate_recognizer/inference_en_v2'
DETECTOR_PATH = PROJECT_ROOT / 'ml/experiments/yolo_license_plate_detector/yolo11n_plate_v1/weights/best.pt'
TEST_V1_DIR = PROJECT_ROOT / 'TEMPORARY/Test V1'
OUTPUT_ROOT = PROJECT_ROOT / 'ml/experiments/anpr_end_to_end_v1'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PLATE_PATTERN = re.compile(r'^[A-Z]{1,3}[0-9]{1,4}[A-Z]{0,3}$')
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
DEVICE = 'gpu:0' if paddle.device.is_compiled_with_cuda() else 'cpu'
YOLO_DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print(f'Paddle device: {DEVICE} | YOLO device: {YOLO_DEVICE}')


## 1. Load detector and fine-tuned recognizer

In [ ]:
for required_path in (RECOGNIZER_DIR / 'inference.yml', DETECTOR_PATH):
    if not required_path.is_file():
        raise FileNotFoundError(required_path)

recognizer = TextRecognition(model_name='en_PP-OCRv5_mobile_rec', model_dir=str(RECOGNIZER_DIR), device=DEVICE)
detector = YOLO(str(DETECTOR_PATH))
print('Fine-tuned PaddleOCR and YOLO detector are ready.')


In [ ]:
def normalize_plate_text(value: str) -> str:
    return re.sub(r'[^A-Z0-9]', '', str(value).upper())

def character_similarity(prediction: str, ground_truth: str) -> float:
    return SequenceMatcher(None, prediction, ground_truth).ratio()

def read_manifest(split: str) -> pd.DataFrame:
    rows = []
    for line in (MANIFEST_ROOT / f'{split}.txt').read_text(encoding='utf-8').splitlines():
        if not line.strip():
            continue
        relative_path, label = line.split('\t', maxsplit=1)
        rows.append({'relative_path': relative_path, 'ground_truth': label, 'image_path': DATA_ROOT / relative_path})
    return pd.DataFrame(rows)

def recognize_image(image: np.ndarray) -> dict:
    started_at = perf_counter()
    outputs = list(recognizer.predict(input=np.ascontiguousarray(image, dtype=np.uint8), batch_size=1))
    latency_ms = (perf_counter() - started_at) * 1000
    payload = outputs[0].json.get('res', outputs[0].json) if outputs else {}
    text = normalize_plate_text(payload.get('rec_text', ''))
    return {'prediction': text, 'confidence': float(payload.get('rec_score', 0.0)), 'latency_ms': latency_ms, 'valid_format': bool(PLATE_PATTERN.fullmatch(text))}

def predict_manifest(split: str, output_path: Path) -> pd.DataFrame:
    if output_path.is_file():
        return pd.read_csv(output_path).fillna('')
    manifest = read_manifest(split)
    rows = []
    for index, row in enumerate(manifest.itertuples(index=False), start=1):
        image = cv2.imread(str(row.image_path))
        result = recognize_image(image)
        result.update({'file': row.relative_path, 'ground_truth': row.ground_truth, 'exact_match': result['prediction'] == row.ground_truth, 'character_similarity': character_similarity(result['prediction'], row.ground_truth)})
        rows.append(result)
        if index % 25 == 0 or index == len(manifest):
            print(f'{split}: {index}/{len(manifest)}')
    frame = pd.DataFrame(rows)
    frame.to_csv(output_path, index=False)
    return frame


## 2. Per-image validation and test predictions

In [ ]:
valid_df = predict_manifest('valid', OUTPUT_ROOT / 'valid_predictions.csv')
test_df = predict_manifest('test', OUTPUT_ROOT / 'test_predictions.csv')

summary_df = pd.DataFrame([
    {'split': 'valid', 'images': len(valid_df), 'exact_accuracy': valid_df['exact_match'].mean(), 'character_similarity': valid_df['character_similarity'].mean(), 'valid_format_rate': valid_df['valid_format'].mean(), 'median_latency_ms': valid_df['latency_ms'].median()},
    {'split': 'test', 'images': len(test_df), 'exact_accuracy': test_df['exact_match'].mean(), 'character_similarity': test_df['character_similarity'].mean(), 'valid_format_rate': test_df['valid_format'].mean(), 'median_latency_ms': test_df['latency_ms'].median()},
])
display(summary_df.style.format({'exact_accuracy': '{:.2%}', 'character_similarity': '{:.2%}', 'valid_format_rate': '{:.2%}', 'median_latency_ms': '{:.1f}'}))
display(test_df[~test_df['exact_match']].sort_values(['character_similarity', 'confidence'])[['file', 'ground_truth', 'prediction', 'confidence', 'character_similarity']].head(30))


## 3. Character confusion analysis

In [ ]:
def align_characters(source: str, target: str) -> list[tuple[str, str]]:
    rows, columns = len(source) + 1, len(target) + 1
    cost = np.zeros((rows, columns), dtype=np.int32)
    cost[:, 0], cost[0, :] = np.arange(rows), np.arange(columns)
    for i in range(1, rows):
        for j in range(1, columns):
            substitution = cost[i - 1, j - 1] + (source[i - 1] != target[j - 1])
            cost[i, j] = min(substitution, cost[i - 1, j] + 1, cost[i, j - 1] + 1)
    pairs, i, j = [], len(source), len(target)
    while i or j:
        if i and j and cost[i, j] == cost[i - 1, j - 1] + (source[i - 1] != target[j - 1]):
            pairs.append((source[i - 1], target[j - 1])); i -= 1; j -= 1
        elif i and cost[i, j] == cost[i - 1, j] + 1:
            pairs.append((source[i - 1], '<DEL>')); i -= 1
        else:
            pairs.append(('<INS>', target[j - 1])); j -= 1
    return list(reversed(pairs))

confusions = []
for row in test_df.itertuples(index=False):
    for expected, predicted in align_characters(str(row.ground_truth), str(row.prediction)):
        if expected != predicted:
            confusions.append({'expected': expected, 'predicted': predicted})
confusion_df = pd.DataFrame(confusions).value_counts(['expected', 'predicted']).rename('count').reset_index()
display(confusion_df.head(30))
confusion_df.to_csv(OUTPUT_ROOT / 'character_confusions.csv', index=False)


## 4. Calibrate the operational decision gate on validation

Only validation data chooses the threshold. `ACCEPT` means the prediction may continue automatically, `REVIEW` requires another frame or operator confirmation, and `REJECT` means no usable plate was read.

In [ ]:
threshold_rows = []
for threshold in np.arange(0.50, 0.99, 0.02):
    accepted = valid_df[(valid_df['confidence'] >= threshold) & valid_df['valid_format']]
    threshold_rows.append({'threshold': threshold, 'coverage': len(accepted) / len(valid_df), 'accepted_accuracy': accepted['exact_match'].mean() if len(accepted) else np.nan, 'accepted_images': len(accepted)})
threshold_df = pd.DataFrame(threshold_rows)
eligible = threshold_df[(threshold_df['accepted_accuracy'] >= 0.90) & (threshold_df['accepted_images'] >= 10)]
ACCEPT_THRESHOLD = float(eligible.sort_values('coverage', ascending=False).iloc[0]['threshold']) if not eligible.empty else 0.90
REVIEW_THRESHOLD = max(0.50, ACCEPT_THRESHOLD - 0.15)
display(threshold_df.style.format({'threshold': '{:.2f}', 'coverage': '{:.2%}', 'accepted_accuracy': '{:.2%}'}))
print(f'ACCEPT_THRESHOLD={ACCEPT_THRESHOLD:.2f} | REVIEW_THRESHOLD={REVIEW_THRESHOLD:.2f}')

def decision_status(text: str, confidence: float) -> str:
    if PLATE_PATTERN.fullmatch(text) and confidence >= ACCEPT_THRESHOLD:
        return 'ACCEPT'
    if text and confidence >= REVIEW_THRESHOLD:
        return 'REVIEW'
    return 'REJECT'

test_df['decision'] = [decision_status(text, confidence) for text, confidence in zip(test_df['prediction'], test_df['confidence'])]
gate_report = test_df.groupby('decision').agg(images=('file', 'count'), exact_accuracy=('exact_match', 'mean'), character_similarity=('character_similarity', 'mean')).reset_index()
display(gate_report.style.format({'exact_accuracy': '{:.2%}', 'character_similarity': '{:.2%}'}))
test_df.to_csv(OUTPUT_ROOT / 'test_predictions_with_gate.csv', index=False)


## 5. YOLO to fine-tuned PaddleOCR pipeline

In [ ]:
def crop_box_with_margin(image: np.ndarray, xyxy: list[float], margin: float = 0.08) -> tuple[np.ndarray, tuple[int, int, int, int]]:
    height, width = image.shape[:2]
    x1, y1, x2, y2 = xyxy
    pad_x, pad_y = (x2 - x1) * margin, (y2 - y1) * margin
    left, top = max(0, int(x1 - pad_x)), max(0, int(y1 - pad_y))
    right, bottom = min(width, int(x2 + pad_x)), min(height, int(y2 + pad_y))
    return image[top:bottom, left:right].copy(), (left, top, right, bottom)

def detect_and_recognize(image: np.ndarray, detection_threshold: float = 0.40) -> tuple[np.ndarray, list[dict]]:
    prediction = detector.predict(image, conf=detection_threshold, imgsz=960, device=YOLO_DEVICE, verbose=False)[0]
    annotated, readings = image.copy(), []
    boxes = prediction.boxes if prediction.boxes is not None else []
    for plate_index, box in enumerate(sorted(boxes, key=lambda item: float(item.xyxy[0][0])), start=1):
        crop, (x1, y1, x2, y2) = crop_box_with_margin(image, box.xyxy[0].tolist())
        if not crop.size:
            continue
        result = recognize_image(crop)
        status = decision_status(result['prediction'], result['confidence'])
        readings.append({'plate_index': plate_index, 'prediction': result['prediction'], 'detection_confidence': float(box.conf[0]), 'ocr_confidence': result['confidence'], 'decision': status, 'ocr_latency_ms': result['latency_ms'], 'crop': crop})
        colors = {'ACCEPT': (0, 170, 0), 'REVIEW': (0, 165, 255), 'REJECT': (0, 0, 220)}
        color = colors[status]
        cv2.rectangle(annotated, (x1, y1), (x2, y2), color, max(2, image.shape[1] // 600))
        label = f'{result["prediction"] or "UNREADABLE"} | {status} | {result["confidence"]:.0%}'
        cv2.putText(annotated, label, (x1, max(25, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2, cv2.LINE_AA)
    return annotated, readings


## 6. Upload a vehicle image

In [ ]:
upload = widgets.FileUpload(accept='image/*', multiple=False, description='Choose image')
run_button = widgets.Button(description='Run ANPR', icon='play', button_style='primary')
upload_output = widgets.Output()

def uploaded_bytes(widget: widgets.FileUpload) -> bytes:
    value = widget.value
    if isinstance(value, dict):
        return bytes(next(iter(value.values()))['content'])
    return bytes(value[0]['content'])

def run_upload(_=None) -> None:
    with upload_output:
        clear_output(wait=True)
        if not upload.value:
            print('Choose an image first.'); return
        image = cv2.imdecode(np.frombuffer(uploaded_bytes(upload), np.uint8), cv2.IMREAD_COLOR)
        annotated, readings = detect_and_recognize(image)
        ok, encoded = cv2.imencode('.jpg', annotated)
        if ok:
            display(DisplayImage(data=encoded.tobytes(), width=1000))
        if readings:
            display(pd.DataFrame([{key: value for key, value in row.items() if key != 'crop'} for row in readings]))
        else:
            print('No plate detected.')

run_button.on_click(run_upload)
display(widgets.VBox([upload, run_button, upload_output]))


## 7. Batch end-to-end test on Test V1

In [ ]:
RUN_TEST_V1 = True
TEST_V1_CSV = OUTPUT_ROOT / 'test_v1_predictions.csv'
if RUN_TEST_V1:
    rows = []
    paths = sorted(path for path in TEST_V1_DIR.iterdir() if path.suffix.lower() in IMAGE_EXTENSIONS)
    for index, image_path in enumerate(paths, start=1):
        image = cv2.imread(str(image_path))
        _, readings = detect_and_recognize(image)
        if not readings:
            rows.append({'file': image_path.name, 'plate_index': 0, 'prediction': '', 'detection_confidence': 0.0, 'ocr_confidence': 0.0, 'decision': 'NO_PLATE_DETECTED', 'ocr_latency_ms': 0.0})
        for reading in readings:
            rows.append({'file': image_path.name, **{key: value for key, value in reading.items() if key != 'crop'}})
        if index % 10 == 0:
            print(f'Test V1: {index}/{len(paths)}')
    test_v1_df = pd.DataFrame(rows)
    test_v1_df.to_csv(TEST_V1_CSV, index=False)
    display(test_v1_df['decision'].value_counts().rename_axis('decision').reset_index(name='count'))
    print(f'Result: {TEST_V1_CSV}')


## 8. Active-learning queue for the next train labels

The queue prioritizes pending train crops with low confidence or invalid-format predictions. It does not modify labels automatically.

In [ ]:
BUILD_ACTIVE_LEARNING_QUEUE = True
ACTIVE_LEARNING_LIMIT = 1000
QUEUE_PATH = DATA_ROOT / 'active_learning_queue.csv'

if BUILD_ACTIVE_LEARNING_QUEUE:
    annotations = pd.read_csv(ANNOTATION_PATH).fillna('')
    pending_pool = annotations[(annotations['split'] == 'train') & (annotations['status'] == 'PENDING')]
    pending = pending_pool.sample(n=min(ACTIVE_LEARNING_LIMIT, len(pending_pool)), random_state=43)
    queue_rows = []
    for index, row in enumerate(pending.itertuples(index=False), start=1):
        image = cv2.imread(str(row.crop_path))
        result = recognize_image(image)
        priority = (1.0 - result['confidence']) + (0.5 if not result['valid_format'] else 0.0)
        queue_rows.append({'sample_id': row.sample_id, 'crop_path': row.crop_path, 'suggestion': result['prediction'], 'confidence': result['confidence'], 'valid_format': result['valid_format'], 'priority_score': priority})
        if index % 100 == 0:
            print(f'Active learning: {index}/{len(pending)}')
    queue_df = pd.DataFrame(queue_rows).sort_values('priority_score', ascending=False)
    queue_df.to_csv(QUEUE_PATH, index=False)
    display(queue_df.head(30))
    print(f'Queue: {QUEUE_PATH}')


## Next decision

Review per-image errors and gate coverage first. If automatic ACCEPT accuracy is adequate, integrate the exported recognizer into the application pipeline. Otherwise label the highest-priority queue samples, retrain as v3, and compare against this fixed v2 test result.